# CNN on CIFAR-10
Three convolutional blocks (32 → 64 → 128 filters) with batch normalisation, dropout and he-normal initialisation; Adam with early stopping and learning-rate reduction callbacks.

**Status:** this run was stopped at epoch 13 of 100 (Colab session limit) with **validation accuracy 76.5% and still rising**. Re-run on a GPU runtime to train to convergence.

In [1]:
# Optional: mount Google Drive to save checkpoints
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import h5py
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, GlobalAveragePooling2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler, ModelCheckpoint
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.constraints import MaxNorm

In [4]:
(X_train, y_train),(X_test, y_test) = cifar10.load_data()
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


X_train shape: (50000, 32, 32, 3)
y_train shape: (50000, 1)
X_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)

In [13]:
from keras import backend as k
k.clear_session()

In [ ]:
num_classes = 10
model = Sequential()

model.add(
    Conv2D(
        filters=32,                    # Number of output filters/channels
        kernel_size=(3, 3),            # Size of the convolution window
        strides=1,                     # Step size of the convolution
        padding='same',                # Padding strategy (same means output size will be made same as input size)
        input_shape=(32, 32, 3),       # Input dimensions (height, width, channels)
        kernel_initializer='he_normal',# Weight initialization method (other options: https://keras.io/api/layers/initializers/)
        activation='relu'              # Activation function (other options: https://keras.io/api/layers/activations/)
    )
)

model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(64,
                 (3, 3),
                 strides=1,
                 padding='same',
                 kernel_initializer='he_normal',
                 activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))

# Second Convolutional Layer in Second Block
model.add(Conv2D(64,
                 (3, 3),
                 strides=1,
                 padding='same',
                 kernel_initializer='he_normal',
                 activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())  # Further reduce spatial dimensions (8x8)


# Third Convolutional Block - Doubled filters (128)
model.add(Conv2D(128,
                 (3, 3),
                 strides=1,
                 padding='same',
                 kernel_initializer='he_normal',
                 activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))

# Second Convolutional Layer in Third Block
model.add(Conv2D(128,
                 (3, 3),
                 strides=1,
                 padding='same',
                 activation='relu'))
model.add(MaxPooling2D())

model.add(GlobalAveragePooling2D())

# Final Dropout before classification
model.add(Dropout(0.2))

# Output layer with softmax activation for 10-class classification
model.add(Dense(num_classes, activation='softmax'))


model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

#filepath to save the best model
filepath="/content/drive/MyDrive/Colab Notebooks/TimesPro/checkpoint.model_GAP.{epoch:02d}.weights.h5"         # when save_weights_only=True

modelcheckpoint= ModelCheckpoint(filepath=filepath, save_weights_only=True,
                                 monitor='val_accuracy', mode='max', save_best_only=True)
#Define callback
callbacks=[modelcheckpoint]


#Training the model
# 320000 training samples, batch size =32 , 320000/32=1000
history = model.fit(
    train_datagen.flow(x_train1, y_train1, batch_size=32),
    validation_data=valid_datagen.flow(x_valid, y_valid, batch_size=32),
    epochs=100,                        # Set a high number of epochs
    callbacks=callbacks                # Include the early stopping callback
)



Model: "sequential_1"


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 16, 16, 64)     │           256 │

 Total params: 280,202 (1.07 MB)


 Trainable params: 279,626 (1.07 MB)


 Non-trainable params: 576 (2.25 KB)


Epoch 1/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 354s 280ms/step - accuracy: 0.4657 - loss: 1.4817 - val_accuracy: 0.5654 - val_loss: 1.2042
Epoch 2/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 330s 264ms/step - accuracy: 0.6043 - loss: 1.1188 - val_accuracy: 0.5804 - val_loss: 1.1994
Epoch 3/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 329s 263ms/step - accuracy: 0.6567 - loss: 0.9789 - val_accuracy: 0.6650 - val_loss: 0.9598
Epoch 4/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 350s 280ms/step - accuracy: 0.6903 - loss: 0.8880 - val_accuracy: 0.6819 - val_loss: 0.9073
Epoch 5/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 327s 262ms/step - accuracy: 0.7129 - loss: 0.8190 - val_accuracy: 0.6168 - val_loss: 1.1138
Epoch 6/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 337s 270ms/step - accuracy: 0.7305 - loss: 0.7799 - val_accuracy: 0.7126 - val_loss: 0.8268
Epoch 7/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 345s 276ms/step - accuracy: 0.7409 - loss: 0.7439 - val_accuracy: 0.7168 - val_loss: 0.8081
Epoch 8/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 326s 261ms/s